# LangSmith token and cost lookup

Uses the lookup functions in `src/ner_review/tokens.py` to show token usage and cost for each LangSmith LLM trace.

`run_id` is the LangSmith trace ID. Change `SINCE_DAYS` to adjust the lookup period.

In [3]:
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langsmith import Client

from src.ner_review.tokens import extract_run_data, query_runs

PROJECT_ROOT = Path.cwd()
# load_dotenv(PROJECT_ROOT / '.env')
# PROJECT_NAME = os.getenv('LANGSMITH_PROJECT')
PROJECT_NAME = 'ner-poc-20261111' # ner-poc-20261112
SINCE_DAYS = 3

print(f'project: {PROJECT_NAME}')
print(f'period: last {SINCE_DAYS} days')

project: ner-poc-20261111
period: last 3 days


In [4]:
client = Client()
project = client.read_project(project_name=PROJECT_NAME)
min_start_time = datetime.now(timezone.utc) - timedelta(days=SINCE_DAYS)
runs = await query_runs(client, project.id, min_start_time)
runs.sort(key=lambda run: run.start_time.timestamp() if run.start_time else 0, reverse=True)

rows = [extract_run_data(run, PROJECT_NAME) for run in runs]
run_frame = pd.DataFrame(rows)

if run_frame.empty:
    print('No LLM traces found.')
else:
    numeric_columns = [
        'input_tokens', 'output_tokens', 'cached_input_tokens',
        'cache_write_tokens', 'total_tokens', 'total_cost',
        'input_cost', 'output_cost', 'duration_seconds',
    ]
    for column in numeric_columns:
        run_frame[column] = pd.to_numeric(run_frame[column], errors='coerce')

    display(run_frame)

,start_time,run_id,project_name,model,status,input_tokens,output_tokens,cached_input_tokens,cache_write_tokens,total_tokens,total_cost,input_cost,output_cost,duration_seconds
0,2026-11-11T11:28:03.551581+00:00,01a1ca5d-6fdf-73f1-842d-78ef15f828b6,ner-poc-20261111,gpt-4.1,success,33201,1452,NaN,NaN,34653,0.035202,0.023586,0.011616,8.016915
1,2026-11-11T11:27:58.602853+00:00,01a1ca5d-5c8a-7e7f-8861-2c54eee2b0f7,ner-poc-20261111,gpt-4.1,success,33146,473,NaN,NaN,33619,0.027260,0.023476,0.003784,4.937741
2,2026-11-11T11:27:53.286576+00:00,01a1ca5d-47c6-757d-93a1-66a71dba8b2c,ner-poc-20261111,gpt-4.1,success,31462,526,NaN,NaN,31988,0.067132,0.062924,0.004208,5.304267
3,2026-11-11T11:27:49.316041+00:00,01a1ca5d-3844-7141-99e4-bf28817f3d83,ner-poc-20261111,gpt-4.1,success,31143,667,NaN,NaN,31810,0.024806,0.019470,0.005336,3.947760
4,2026-11-11T11:27:45.330448+00:00,01a1ca5d-28b2-7cd0-a9fa-f00940156b47,ner-poc-20261111,gpt-4.1,success,33192,269,NaN,NaN,33461,0.068536,0.066384,0.002152,3.933115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2026-11-11T09:17:34.408007+00:00,01a1c9e5-f948-70bb-933f-136ca5c1369d,ner-poc-20261111,gpt-5.1,success,31115,769,NaN,NaN,31884,0.013896,0.006206,0.007690,9.217151
96,2026-11-11T09:17:28.757380+00:00,01a1c9e5-e335-7d0d-af46-d129c42fac8b,ner-poc-20261111,gpt-5.1,success,31723,455,NaN,NaN,32178,0.011516,0.006966,0.004550,5.643394
97,2026-11-11T09:17:24.780739+00:00,01a1c9e5-d3ac-7dd4-b9c0-58fcbc654b27,ner-poc-20261111,gpt-5.1,success,32846,131,NaN,NaN,32977,0.009680,0.008370,0.001310,3.968164
98,2026-11-11T09:17:09.909035+00:00,01a1c9e5-9995-7efb-8046-c08f17336cfd,ner-poc-20261111,gpt-5.1,success,34913,1174,NaN,NaN,36087,0.022693,0.010953,0.011740,14.871704


In [8]:
run_frame[run_frame["model"] == 'claude-haiku-5.5']

,start_time,run_id,project_name,model,status,input_tokens,output_tokens,cached_input_tokens,cache_write_tokens,total_tokens,total_cost,input_cost,output_cost,duration_seconds


In [5]:
if not run_frame.empty:
    summary = pd.DataFrame({
        'trace_count': [len(run_frame)],
        'total_input_tokens': [run_frame['input_tokens'].sum()],
        'total_output_tokens': [run_frame['output_tokens'].sum()],
        'total_tokens': [run_frame['total_tokens'].sum()],
        'total_cost': [run_frame['total_cost'].sum()],
    })
    display(summary)

,trace_count,total_input_tokens,total_output_tokens,total_tokens,total_cost
0,100,3333550,46242,3379792,3.240566
